[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/02_synthetic_qa_generation.ipynb)

# Step 2 — Synthetic Q&A Generation Strategies

Compare how a **teacher LLM** generates policy Q&A using different prompting strategies, and save the raw outputs as **training** data for the distillation pipeline (filtered in Step 3, used to fine-tune the small model later).

## Scope: training data only
This notebook operates exclusively on **TRAIN-split paragraphs**. It is *not* where the evaluation/test set is built — that happens in Step 1 via a separate `generate_test_qa_batch` path that targets specific small-model `FailureMode`s (format non-compliance, domain vocabulary drift, refusal calibration, multi-constraint collapse).

Because the `QASample` schema is shared across train and test samples (one JSONL shape, one set of loaders/filters), training samples still carry a `failure_mode` field — but it will be `None` here, since failure-mode targeting is a test-set concern. Don't be surprised when cell 8 prints `Failure mode: None` for every strategy.

## Learning objectives
- Zero-shot, one-shot, few-shot, and topic-controlled generation
- Understand distillation: strong model → synthetic training data → small model
- See why train-side `failure_mode` is `None` while test-side samples set it explicitly

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    PARAGRAPHS_PATH,
    SYNTHETIC_RAW_PATH,
    Paragraph,
    ParagraphSplit,
    QASample,
    compare_generation_strategies,
    create_teacher_client,
    generate_raw_synthetic_corpus,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [2]:
import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any earlier basicConfig from other libs
)

## 1. Load TRAIN paragraphs only

Use paragraphs saved in Step 1 where `split == train`.

In [3]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
console.print(f"Train paragraphs available: {len(train_paragraphs)}")

para = train_paragraphs[1]

table = Table(title="Train Paragraph [1]", show_lines=True)
table.add_column("Field", style="bold cyan")
table.add_column("Value", style="white")

table.add_row("doc_id", str(para.doc_id))
table.add_row("para_id", str(para.para_id))
table.add_row("role", str(para.role))
table.add_row("split", str(para.split))
table.add_row("index", str(para.index))
table.add_row("text", para.text if len(para.text) < 500 else para.text[:500] + " ...")

console = Console()
console.print(Panel(table, title="Train Paragraph Overview"))

Train paragraphs available: 36

╭─────────────────────────────────────────── Train Paragraph Overview ────────────────────────────────────────────╮
│                                               Train Paragraph [1]                                               │
│ ┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Field   ┃ Value                                                                                             ┃ │
│ ┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ doc_id  │ cfpb_credit_card_agreement                                                                        │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ para_id │ cfpb_credit_card_agreement::p0001                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ role    │ policy_dense                                                                                      │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ split   │ train                                                                                             │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ index   │ 1                                                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ text    │ 1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a     │ │
│ │         │ line of credit for you.                                                                           │ │
│ │         │ You agree that your credit limit is the maximum amount (purchases, cash advances, finance         │ │
│ │         │ charges, plus "other                                                                              │ │
│ │         │ charges") which you will have outstanding on your account at any time. Unless disclosed           │ │
│ │         │ otherwise, the Credit Union will                                                                  │ │
│ │         │ not allow advances over the credit limit. If the Credit Union has a program whereby it allows     │ │
│ │         │ payment of advances that                                                                          │ │
│ │         │ exceed your credit limit, subje ...                                                               │ │
│ └─────────┴───────────────────────────────────────────────────────────────────────────────────────────────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 2. Compare generation strategies on the same paragraph

Each strategy is called **without** a `failure_mode`, so the printed `Failure mode: None` is expected — that field is only populated by the test-set construction path in Step 1. Here, what matters is the question/answer shape each strategy produces from the same source paragraph.

In [4]:
teacher = create_teacher_client()
demo_paragraph = train_paragraphs[1]

# One-shot: a single in-context Q&A that shows the teacher the desired format.
# If omitted, compare_generation_strategies uses this same grace-period seed.
one_shot_example = QASample(
    id="one-shot-seed",
    question="What is the grace period for new purchases?",
    gold_answer="The grace period ends 21 days after the close of the billing cycle.",
    doc_id=demo_paragraph.doc_id,
    para_id=demo_paragraph.para_id,
    context=demo_paragraph.text,
    instruction="Answer using the source passage. Respond in one sentence.",
)

# Few-shot: several examples so the teacher can copy format and vary the angle.
# If omitted, the helper falls back to [one_shot_example],
# so few-shot collapses to one-shot.
few_shot_examples = [
    one_shot_example,
    QASample(
        id="few-shot-apr",
        question="What is the purchase APR, and when can the issuer raise it?",
        gold_answer="The purchase APR is 24.99%. The issuer may raise it if you make a late payment or go over your credit limit.",
        doc_id=demo_paragraph.doc_id,
        para_id=demo_paragraph.para_id,
        context=demo_paragraph.text,
        instruction="Answer using the source passage. Respond in one sentence.",
    ),
    QASample(
        id="few-shot-late-fee",
        question="How much is the late payment fee?",
        gold_answer="The late payment fee is $40 if you do not pay at least the minimum amount due by the due date.",
        doc_id=demo_paragraph.doc_id,
        para_id=demo_paragraph.para_id,
        context=demo_paragraph.text,
        instruction="Answer using the source passage. Respond in one sentence.",
    ),
]

strategy_outputs = compare_generation_strategies(
    teacher,
    demo_paragraph,
    seed_example=one_shot_example,
    few_shot_examples=few_shot_examples,
    topic_controlled_topic="client fees",
)
for strategy, sample in strategy_outputs.items():
    from rich.text import Text

    header = Text(f"{strategy}", style="bold green")
    question = Text(f"Q: {sample.question}", style="cyan")
    answer = Text(f"A: {sample.gold_answer}", style="magenta")
    failure_mode = Text(f"Failure mode: {sample.failure_mode}", style="yellow")

    console.print(
        Panel(
            Text.assemble(header, "\n", question, "\n", answer, "\n", failure_mode),
            title=f"Generation Strategy: {strategy}",
            border_style="green",
        )
    )

2026-08-25 18:16:03,322 INFO aieng.syn_data.text.generation: Payload: {'question': 'Under what conditions does an oral stop payment request for a convenience check remain effective beyond 14 days, and how long does a written stop payment order remain in effect?', 'gold_answer': 'An oral stop payment request will expire after 14 days unless you confirm the request in writing within that time. Once in writing, the stop payment order is effective for six months (and can be renewed in writing for additional six-month periods).'}


╭──────────────────────────────────────── Generation Strategy: zero_shot ─────────────────────────────────────────╮
│ zero_shot                                                                                                       │
│ Q: Under what conditions does an oral stop payment request for a convenience check remain effective beyond 14   │
│ days, and how long does a written stop payment order remain in effect?                                          │
│ A: An oral stop payment request will expire after 14 days unless you confirm the request in writing within that │
│ time. Once in writing, the stop payment order is effective for six months (and can be renewed in writing for    │
│ additional six-month periods).                                                                                  │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Generation Strategy: one_shot ─────────────────────────────────────────╮
│ one_shot                                                                                                        │
│ Q: Under what condition will an oral stop payment request for a convenience check remain in effect for longer   │
│ than 14 days?                                                                                                   │
│ A: An oral stop payment request will remain in effect for longer than 14 days only if you confirm your request  │
│ in writing within that 14-day timeframe.                                                                        │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Generation Strategy: few_shot ─────────────────────────────────────────╮
│ few_shot                                                                                                        │
│ Q: Under what conditions will an oral stop payment request for a convenience check expire, and how long are     │
│ written stop payment orders effective?                                                                          │
│ A: An oral stop payment request will expire after 14 days unless confirmed in writing within that time, while   │
│ written stop payment orders are effective for six months.                                                       │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── Generation Strategy: topic_controlled ─────────────────────────────────────╮
│ topic_controlled                                                                                                │
│ Q: Under what condition does a client agree to pay a fee related to convenience checks, and what must they do   │
│ to keep an oral request for this action active beyond 14 days?                                                  │
│ A: A client agrees to pay a fee when they request to stop the payment of a convenience check drawn on their     │
│ account. To keep an oral stop payment request active beyond 14 days, the client must confirm the request in     │
│ writing within that 14-day timeframe.                                                                           │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Quick pick guide

Start simple → zero-shot 

Want consistent format → one/few-shot 

Want variety from long paragraphs → topic-controlled 

## 3. Save raw synthetic samples

Store outputs for quality filtering in Step 3. Each train paragraph contributes one zero-shot, one-shot, and few-shot sample, plus up to `questions_per_para` topic-controlled samples from topics extracted from that paragraph. The comparison demo above still shows one sample per strategy.

In [5]:
raw_samples = generate_raw_synthetic_corpus(
    teacher,
    train_paragraphs,
    max_paragraphs=len(train_paragraphs),
    questions_per_para=2,
)
console.print(f"Raw synthetic samples: {len(raw_samples)}")

save_typed_jsonl(
    SYNTHETIC_RAW_PATH,
    raw_samples,
    to_dict=lambda sample: sample.to_dict(),
)
SYNTHETIC_RAW_PATH

2026-08-25 18:17:02,211 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement?', 'gold_answer': 'The Account Opening Disclosure (also referred to as the "Disclosure" or "Credit Card Account Opening Disclosure") is incorporated into and is part of the Agreement.'}
2026-08-25 18:17:03,749 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement itself?', 'gold_answer': 'The Account Opening Disclosure (referred to as "Disclosure") is incorporated into and is part of the Agreement.'}
2026-08-25 18:17:12,455 INFO aieng.syn_data.text.generation: Payload: {'question': 'Under what conditions does an oral stop payment request for a convenience check remain effective beyond 14 days, and how long does a written stop payment order remain i

Raw synthetic samples: 180

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/synthetic_raw.jsonl')